<!-- 72/19 Hold-Out Validation with Nested CV Inside Training Set  -->

For each seed: 91 subjects are split into 72 training/development subjects and 19 untouched held-out test subjects. Inside only the 72 training subjects, a stratified 10-fold outer CV is performed; inside each outer-training fold, LOOCV GridSearch selects the best k. After this nested CV, the consensus k is selected, the final model is trained on all 72 subjects, and evaluated once on the untouched 19 subjects.

Classifiers: RF, SVM_Lin, LR, LDA, XGBoost.

In [2]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: xgboost in c:\users\welcome\anaconda3\lib\site-packages (2.1.4)




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:

# =============================================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split, StratifiedKFold, LeaveOneOut, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost is not installed. Run: pip install xgboost")

DATA_PATH = "E:/cleaned_csv_reduced_430321.csv"
TARGET_COL = "MDD"
DROP_COLS = ["MDD", "ID", "Species"]

K_VALUES = [10, 15, 20, 25, 30]
CLF_NAMES = ["RF", "SVM_Lin", "LR", "LDA", "XGBoost"]
RANDOM_SEEDS = [42, 0, 1, 7, 99]

TEST_SIZE = 0.20
N_OUTER_FOLDS = 10
N_BOOTSTRAP = 1000
SEED = 42

METRICS = ["accuracy", "precision", "sensitivity", "specificity", "f1", "roc_auc"]

print("=" * 100)
print("72/19 HOLD-OUT VALIDATION WITH NESTED CV INSIDE TRAINING SET")
print("=" * 100)
print(f"Seeds           : {RANDOM_SEEDS}")
print(f"Hold-out split  : 80/20 stratified split per seed")
print(f"Training CV     : Stratified 10-fold outer CV inside 72 subjects")
print(f"Inner selection : LOOCV GridSearch inside each outer training fold")
print(f"Classifiers     : {CLF_NAMES}")
print("=" * 100)


72/19 HOLD-OUT VALIDATION WITH NESTED CV INSIDE TRAINING SET
Seeds           : [42, 0, 1, 7, 99]
Hold-out split  : 80/20 stratified split per seed
Training CV     : Stratified 10-fold outer CV inside 72 subjects
Inner selection : LOOCV GridSearch inside each outer training fold
Classifiers     : ['RF', 'SVM_Lin', 'LR', 'LDA', 'XGBoost']


In [4]:

# =============================================================================
# CELL 2 — DATA LOADING
# =============================================================================

df = pd.read_csv(DATA_PATH)

labels = df[TARGET_COL]
encoder = LabelEncoder()
Y = encoder.fit_transform(labels).astype(int)

X_df = df.drop(columns=DROP_COLS, errors="ignore")
feature_names = list(X_df.columns)
X = X_df.values.astype(float)

print("=" * 80)
print("DATA OVERVIEW")
print("=" * 80)
print(f"X shape       : {X.shape}")
print(f"Y shape       : {Y.shape}")
print(f"Feature count : {len(feature_names)}")
print(f"Class mapping : {dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))}")
print(f"Class counts  : {dict(Counter(Y.tolist()))}")

assert X.shape[0] == len(Y), "X and Y row count mismatch"
assert not np.isnan(X).any(), "X contains NaN values"
assert not np.isinf(X).any(), "X contains infinite values"
assert len(np.unique(Y)) == 2, "Y must be binary"
assert X.var(axis=0).min() > 0, "Some features have zero variance"

print("\nIntegrity checks passed.")


DATA OVERVIEW
X shape       : (91, 314)
Y shape       : (91,)
Feature count : 314
Class mapping : {0: 0, 1: 1}
Class counts  : {1: 50, 0: 41}

Integrity checks passed.


In [5]:

# =============================================================================
# CELL 3 — METRIC FUNCTIONS
# =============================================================================

def compute_metrics(y_true, y_pred, y_prob):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "confusion_matrix": cm,
        "pred_dist": dict(Counter(y_pred.tolist()))
    }


def bootstrap_ci(y_true, y_pred, y_prob, n_bootstrap=N_BOOTSTRAP):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    results = {m: [] for m in METRICS}
    n = len(y_true)
    rng = np.random.default_rng(42)

    for _ in range(n_bootstrap):
        idx = resample(
            np.arange(n),
            replace=True,
            stratify=y_true,
            random_state=int(rng.integers(0, 2**31 - 1))
        )

        yt = y_true[idx]
        yp = y_pred[idx]
        yb = y_prob[idx]

        if len(np.unique(yt)) < 2:
            continue

        cm = confusion_matrix(yt, yp, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        results["accuracy"].append(accuracy_score(yt, yp))
        results["precision"].append(precision_score(yt, yp, zero_division=0))
        results["sensitivity"].append(recall_score(yt, yp, zero_division=0))
        results["specificity"].append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
        results["f1"].append(f1_score(yt, yp, zero_division=0))
        results["roc_auc"].append(roc_auc_score(yt, yb))

    return {
        m: (
            float(np.mean(results[m])),
            float(np.percentile(results[m], 2.5)),
            float(np.percentile(results[m], 97.5))
        )
        for m in METRICS
    }


In [6]:

# =============================================================================
# CELL 4 — PIPELINE FACTORIES
# =============================================================================

def make_classifier(clf_name):
    if clf_name == "RF":
        return RandomForestClassifier(
            n_estimators=200,
            max_depth=4,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        )

    if clf_name == "SVM_Lin":
        return SVC(
            kernel="linear",
            C=1,
            class_weight="balanced",
            probability=True,
            random_state=SEED
        )

    if clf_name == "LR":
        return LogisticRegression(
            C=1,
            max_iter=3000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=SEED
        )

    if clf_name == "LDA":
        return LinearDiscriminantAnalysis(
            solver="lsqr",
            shrinkage="auto"
        )

    if clf_name == "XGBoost":
        if not HAS_XGBOOST:
            raise ImportError("XGBoost is not installed. Run: pip install xgboost")
        return XGBClassifier(
            n_estimators=50,
            max_depth=2,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=5,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1
        )

    raise ValueError(f"Unknown classifier: {clf_name}")


def make_pipeline_fixed_k(clf_name, k):
    return Pipeline([
        ("scaler", RobustScaler()),
        ("selector", SelectKBest(score_func=f_classif, k=k)),
        ("clf", make_classifier(clf_name))
    ])


def make_inner_gridsearch_pipeline(clf_name):
    pipe = Pipeline([
        ("scaler", RobustScaler()),
        ("selector", SelectKBest(score_func=f_classif, k=K_VALUES[0])),
        ("clf", make_classifier(clf_name))
    ])

    param_grid = {"selector__k": K_VALUES}

    return GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=LeaveOneOut(),
        scoring="accuracy",
        refit=True,
        n_jobs=-1,
        error_score=0.0
    )

print("Pipeline factories created.")


Pipeline factories created.


In [7]:

# =============================================================================
# CELL 5 — NESTED CV INSIDE 72-SUBJECT TRAINING SET
# =============================================================================

def run_nested_cv_on_training_set(X_train, Y_train, seed):
    """
    Runs nested CV only on the 72-subject training/development partition.

    Structure:
    Stratified 10-fold outer CV
        -> Inner LOOCV GridSearch for best k
        -> Refit on outer-training fold
        -> Test on outer-test fold
    """

    outer_cv = StratifiedKFold(
        n_splits=N_OUTER_FOLDS,
        shuffle=True,
        random_state=seed
    )

    nested_metrics = {}
    best_k_log = {}
    nested_preds = {}
    nested_probs = {}

    for clf_name in CLF_NAMES:
        y_pred_all = np.zeros(len(Y_train), dtype=int)
        y_prob_all = np.zeros(len(Y_train), dtype=float)
        fold_best_ks = []

        print(f"\nNested CV inside 72 training subjects | Classifier: {clf_name}")

        for fold_idx, (tr_idx, val_idx) in enumerate(outer_cv.split(X_train, Y_train), 1):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            Y_tr, Y_val = Y_train[tr_idx], Y_train[val_idx]

            gs = make_inner_gridsearch_pipeline(clf_name)
            gs.fit(X_tr, Y_tr)

            best_k = gs.best_params_["selector__k"]
            fold_best_ks.append(best_k)

            y_pred = gs.predict(X_val)
            y_prob = gs.predict_proba(X_val)[:, 1]

            y_pred_all[val_idx] = y_pred
            y_prob_all[val_idx] = y_prob

            print(
                f"  Outer fold {fold_idx:02d}/{N_OUTER_FOLDS} | "
                f"best_k={best_k:<2} | "
                f"val_dist={dict(Counter(Y_val.tolist()))} | "
                f"pred_dist={dict(Counter(y_pred.tolist()))}"
            )

        nested_metrics[clf_name] = compute_metrics(Y_train, y_pred_all, y_prob_all)
        best_k_log[clf_name] = fold_best_ks
        nested_preds[clf_name] = y_pred_all.copy()
        nested_probs[clf_name] = y_prob_all.copy()

        m = nested_metrics[clf_name]
        print(
            f"  Nested-CV summary on 72 | "
            f"Acc={m['accuracy']*100:6.2f}% | "
            f"AUC={m['roc_auc']*100:6.2f}% | "
            f"F1={m['f1']*100:6.2f}%"
        )

    return nested_metrics, best_k_log, nested_preds, nested_probs


def get_consensus_k_from_nested(best_k_log, clf_name):
    return Counter(best_k_log[clf_name]).most_common(1)[0][0]


In [8]:

# =============================================================================
# CELL 6 — FINAL HELD-OUT TEST EVALUATION ON 19 SUBJECTS
# =============================================================================

def evaluate_heldout_test_after_nested_cv(X_train, Y_train, X_test, Y_test, best_k_log):
    """
    After nested CV is completed inside the 72-subject training set,
    this trains a final fixed-k model on all 72 subjects and evaluates
    it once on the untouched 19-subject held-out test set.
    """

    test_metrics = {}
    test_ci = {}
    final_k = {}

    for clf_name in CLF_NAMES:
        consensus_k = get_consensus_k_from_nested(best_k_log, clf_name)
        final_k[clf_name] = consensus_k

        final_pipe = make_pipeline_fixed_k(clf_name, consensus_k)
        final_pipe.fit(X_train, Y_train)

        y_pred = final_pipe.predict(X_test)
        y_prob = final_pipe.predict_proba(X_test)[:, 1]

        test_metrics[clf_name] = compute_metrics(Y_test, y_pred, y_prob)
        test_ci[clf_name] = bootstrap_ci(Y_test, y_pred, y_prob)

        m = test_metrics[clf_name]
        print(
            f"{clf_name:<10} | consensus_k={consensus_k:<2} | "
            f"TEST Acc={m['accuracy']*100:6.2f}% | "
            f"Prec={m['precision']*100:6.2f}% | "
            f"Sens={m['sensitivity']*100:6.2f}% | "
            f"Spec={m['specificity']*100:6.2f}% | "
            f"F1={m['f1']*100:6.2f}% | "
            f"AUC={m['roc_auc']*100:6.2f}% | "
            f"TP={m['TP']} TN={m['TN']} FP={m['FP']} FN={m['FN']}"
        )

    return test_metrics, test_ci, final_k


In [9]:

# =============================================================================
# CELL 7 — MAIN LOOP: FIVE SEEDS, 72/19 SPLIT, NESTED CV ON 72, TEST ON 19
# =============================================================================

all_seed_results = []

for seed in RANDOM_SEEDS:
    print("\n" + "#" * 100)
    print(f"SEED = {seed}")
    print("#" * 100)

    X_train, X_test, Y_train, Y_test = train_test_split(
        X,
        Y,
        test_size=TEST_SIZE,
        stratify=Y,
        random_state=seed
    )

    print(f"Training/development set : {len(Y_train)} subjects | Distribution: {dict(Counter(Y_train.tolist()))}")
    print(f"Held-out test set        : {len(Y_test)} subjects | Distribution: {dict(Counter(Y_test.tolist()))}")

    nested_metrics, best_k_log, nested_preds, nested_probs = run_nested_cv_on_training_set(
        X_train=X_train,
        Y_train=Y_train,
        seed=seed
    )

    print("\nFinal held-out test evaluation on untouched 19 subjects")
    print("-" * 100)

    test_metrics, test_ci, final_k = evaluate_heldout_test_after_nested_cv(
        X_train=X_train,
        Y_train=Y_train,
        X_test=X_test,
        Y_test=Y_test,
        best_k_log=best_k_log
    )

    all_seed_results.append({
        "seed": seed,
        "Y_train": Y_train.copy(),
        "Y_test": Y_test.copy(),
        "nested_metrics": nested_metrics,
        "best_k_log": best_k_log,
        "nested_preds": nested_preds,
        "nested_probs": nested_probs,
        "test_metrics": test_metrics,
        "test_ci": test_ci,
        "final_k": final_k
    })

print("\n" + "=" * 100)
print("ALL FIVE SEEDS COMPLETED")
print("=" * 100)



####################################################################################################
SEED = 42
####################################################################################################
Training/development set : 72 subjects | Distribution: {1: 40, 0: 32}
Held-out test set        : 19 subjects | Distribution: {1: 10, 0: 9}

Nested CV inside 72 training subjects | Classifier: RF
  Outer fold 01/10 | best_k=30 | val_dist={0: 4, 1: 4} | pred_dist={0: 3, 1: 5}
  Outer fold 02/10 | best_k=10 | val_dist={0: 4, 1: 4} | pred_dist={0: 7, 1: 1}
  Outer fold 03/10 | best_k=20 | val_dist={0: 3, 1: 4} | pred_dist={1: 4, 0: 3}
  Outer fold 04/10 | best_k=15 | val_dist={1: 4, 0: 3} | pred_dist={1: 3, 0: 4}
  Outer fold 05/10 | best_k=30 | val_dist={1: 4, 0: 3} | pred_dist={1: 4, 0: 3}
  Outer fold 06/10 | best_k=15 | val_dist={0: 3, 1: 4} | pred_dist={1: 4, 0: 3}
  Outer fold 07/10 | best_k=10 | val_dist={1: 4, 0: 3} | pred_dist={1: 4, 0: 3}
  Outer fold 08/10 | best_k=30 |

In [10]:

# =============================================================================
# CELL 8 — SUMMARY TABLES
# =============================================================================

def print_metric_summary(all_results, result_key, title):
    W = 135
    print("=" * W)
    print(title)
    print("=" * W)

    header = (
        f"{'Classifier':<12}"
        f"{'Accuracy %':>17}"
        f"{'Precision %':>17}"
        f"{'Sensitivity %':>17}"
        f"{'Specificity %':>17}"
        f"{'F1 %':>17}"
        f"{'AUC %':>17}"
    )

    print(header)
    print("-" * W)

    summary = {}

    for clf_name in CLF_NAMES:
        vals = {m: [] for m in METRICS}

        for res in all_results:
            for m in METRICS:
                vals[m].append(res[result_key][clf_name][m])

        summary[clf_name] = {
            m: (float(np.mean(vals[m])), float(np.std(vals[m], ddof=1)))
            for m in METRICS
        }

        def fmt(metric):
            mean, sd = summary[clf_name][metric]
            return f"{mean*100:6.2f} ± {sd*100:5.2f}"

        print(
            f"{clf_name:<12}"
            f"{fmt('accuracy'):>17}"
            f"{fmt('precision'):>17}"
            f"{fmt('sensitivity'):>17}"
            f"{fmt('specificity'):>17}"
            f"{fmt('f1'):>17}"
            f"{fmt('roc_auc'):>17}"
        )

    print("=" * W)
    return summary


heldout_test_summary = print_metric_summary(
    all_seed_results,
    result_key="test_metrics",
    title="FINAL HELD-OUT TEST PERFORMANCE — MEAN ± SD ACROSS FIVE SEEDS"
)

nested_training_summary = print_metric_summary(
    all_seed_results,
    result_key="nested_metrics",
    title="NESTED CV PERFORMANCE INSIDE 72-SUBJECT TRAINING SET — MEAN ± SD ACROSS FIVE SEEDS"
)


FINAL HELD-OUT TEST PERFORMANCE — MEAN ± SD ACROSS FIVE SEEDS
Classifier         Accuracy %      Precision %    Sensitivity %    Specificity %             F1 %            AUC %
---------------------------------------------------------------------------------------------------------------------------------------
RF              57.89 ± 15.34    60.54 ± 19.90    50.00 ± 23.45    66.67 ± 13.61    53.95 ± 21.24    62.89 ±  8.70
SVM_Lin         66.32 ± 10.26    75.72 ±  9.02    52.00 ± 21.68    82.22 ±  9.94    59.74 ± 18.79    71.11 ±  5.61
LR              64.21 ± 12.57    69.72 ± 14.04    54.00 ± 18.17    75.56 ±  9.30    60.42 ± 16.72    63.11 ±  9.01
LDA             63.16 ±  8.32    67.53 ±  9.29    60.00 ± 12.25    66.67 ± 15.71    62.83 ±  9.28    62.67 ±  8.04
XGBoost         58.95 ± 12.57    59.66 ± 13.23    66.00 ± 20.74    51.11 ± 18.59    61.85 ± 15.71    57.11 ± 20.41
NESTED CV PERFORMANCE INSIDE 72-SUBJECT TRAINING SET — MEAN ± SD ACROSS FIVE SEEDS
Classifier         Accuracy %

In [11]:

# =============================================================================
# CELL 9 — BOOTSTRAP CI, FINAL k, AND CONFUSION MATRIX SUMMARY
# =============================================================================

def print_heldout_bootstrap_ci_summary(all_results):
    W = 110
    print("=" * W)
    print("HELD-OUT TEST BOOTSTRAP 95% CI — AVERAGED ACROSS FIVE SEEDS")
    print("Format: Mean% [Lower% — Upper%]")
    print("=" * W)

    for clf_name in CLF_NAMES:
        print("\n" + clf_name)
        print("-" * 80)

        means = {m: [] for m in METRICS}
        lows = {m: [] for m in METRICS}
        highs = {m: [] for m in METRICS}

        for res in all_results:
            for m in METRICS:
                mean, low, high = res["test_ci"][clf_name][m]
                means[m].append(mean)
                lows[m].append(low)
                highs[m].append(high)

        for m in METRICS:
            print(
                f"{m:<14} "
                f"{np.mean(means[m])*100:6.2f}% "
                f"[{np.mean(lows[m])*100:6.2f}% — {np.mean(highs[m])*100:6.2f}%]"
            )

    print("=" * W)


def print_final_k_table(all_results):
    W = 110
    print("=" * W)
    print("FINAL CONSENSUS k USED FOR HELD-OUT TESTING")
    print("Consensus k = most frequent k selected across the 10 nested-CV outer folds inside 72 training subjects.")
    print("=" * W)

    header = f"{'Classifier':<12}"
    for res in all_results:
        header += f"{'seed=' + str(res['seed']):>12}"
    header += f"{'Most common':>15}"

    print(header)
    print("-" * W)

    for clf_name in CLF_NAMES:
        row = f"{clf_name:<12}"
        ks = []

        for res in all_results:
            k = res["final_k"][clf_name]
            ks.append(k)
            row += f"{'k=' + str(k):>12}"

        most_common = Counter(ks).most_common(1)[0][0]
        row += f"{'k=' + str(most_common):>15}"
        print(row)

    print("=" * W)


def print_heldout_confusion_summary(all_results):
    W = 115
    print("=" * W)
    print("HELD-OUT TEST CONFUSION MATRIX SUMMARY — AVERAGED ACROSS FIVE SEEDS")
    print("TP=True MDD | TN=True HC | FP=HC predicted as MDD | FN=MDD predicted as HC")
    print("=" * W)

    print(
        f"{'Classifier':<12}"
        f"{'TP':>10}"
        f"{'TN':>10}"
        f"{'FP':>10}"
        f"{'FN':>10}"
        f"{'Accuracy %':>15}"
        f"{'Sensitivity %':>15}"
        f"{'Specificity %':>15}"
    )

    print("-" * W)

    for clf_name in CLF_NAMES:
        tp, tn, fp, fn = [], [], [], []
        acc, sens, spec = [], [], []

        for res in all_results:
            m = res["test_metrics"][clf_name]
            tp.append(m["TP"])
            tn.append(m["TN"])
            fp.append(m["FP"])
            fn.append(m["FN"])
            acc.append(m["accuracy"])
            sens.append(m["sensitivity"])
            spec.append(m["specificity"])

        print(
            f"{clf_name:<12}"
            f"{np.mean(tp):>10.1f}"
            f"{np.mean(tn):>10.1f}"
            f"{np.mean(fp):>10.1f}"
            f"{np.mean(fn):>10.1f}"
            f"{np.mean(acc)*100:>15.2f}"
            f"{np.mean(sens)*100:>15.2f}"
            f"{np.mean(spec)*100:>15.2f}"
        )

    print("=" * W)


print_heldout_bootstrap_ci_summary(all_seed_results)
print_final_k_table(all_seed_results)
print_heldout_confusion_summary(all_seed_results)


HELD-OUT TEST BOOTSTRAP 95% CI — AVERAGED ACROSS FIVE SEEDS
Format: Mean% [Lower% — Upper%]

RF
--------------------------------------------------------------------------------
accuracy        58.05% [ 37.89% —  77.89%]
precision       60.99% [ 31.39% —  90.86%]
sensitivity     50.03% [ 22.00% —  78.00%]
specificity     66.95% [ 35.56% —  95.56%]
f1              53.37% [ 26.87% —  76.26%]
roc_auc         63.13% [ 34.44% —  87.78%]

SVM_Lin
--------------------------------------------------------------------------------
accuracy        66.57% [ 47.37% —  84.21%]
precision       76.26% [ 41.02% — 100.00%]
sensitivity     51.97% [ 24.00% —  80.00%]
specificity     82.78% [ 57.78% — 100.00%]
f1              58.96% [ 31.06% —  82.18%]
roc_auc         71.27% [ 44.44% —  92.90%]

LR
--------------------------------------------------------------------------------
accuracy        64.43% [ 44.21% —  84.21%]
precision       70.73% [ 41.59% — 100.00%]
sensitivity     53.87% [ 26.00% —  82.00%]
spe